# idQ: a short demo

Use `idQ` to check the algebraic condition for a binary $J\times K$ $Q$-matrix under the **conjunctive operator**. The number of columns $K$ is fixed, and $Q$ is identified up to column permutations. This notebook demonstrates matrix input, the returned decision, counterexample certificates, basis reduction, and a small reproducible experiment.

The decision concerns the manuscript's algebraic condition. It does **not**, by itself, establish joint identifiability of the slip, guessing, and latent-proportion parameters in a noisy model.

Glucose 4.2 is the default SAT solver. These examples are deliberately small; run all cells from top to bottom.

## Setup

From the top-level `idQ` folder, run these commands in a terminal once:

```bash
python -m pip install -e ".[notebook]"
python -m jupyter lab notebooks/idQ_demo.ipynb
```

Select the Python environment in which you installed `idQ`. The notebook does not install packages or submit cluster jobs. Every example matrix has binary entries and nonzero rows, as required by the manuscript's standing convention.

In [1]:
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

from idQ import identify, reduce_to_basis, reconstruct_from_basis
from idQ.utils import boolean_product, equivalent_up_to_column_permutation
from idQ.experiments.common import sample_bernoulli_q


def show_result(result):
    """Display the main fields; the complete result remains available."""
    display(Markdown(
        f"**Identifiable:** `{result.identifiable}`  \n"
        f"**Reason:** `{result.branch_label}`  \n"
        f"**Basis shape:** `{result.basis.shape}`  \n"
        f"**SAT solver used:** `{result.solver_name}`  \n"
        f"**Cardinality encoding:** `{result.sat_cardinality_encoding}`"
    ))

## 1. A first call

Pass a two-dimensional NumPy array or a nested list to `identify`. For $Q=I_3$, preprocessing immediately certifies the algebraic condition, so no SAT call is needed. A `None` solver name here means that preprocessing finished the calculation.

In [2]:
Q_identity = np.eye(3, dtype=int)
result_identity = identify(Q_identity)

show_result(result_identity)
assert result_identity.identifiable
assert result_identity.solver_name is None

**Identifiable:** `True`  
**Reason:** `identity_submatrix_identifiable`  
**Basis shape:** `(3, 3)`  
**SAT solver used:** `None`  
**Cardinality encoding:** `None`

## 2. An identifiable matrix with no pure nodes

A pure node has a row containing exactly one 1. Every row below contains two 1s. This example reaches SAT, and Glucose proves that no nonequivalent factorization exists.

`identifiable=True` is the conclusion about $Q$. An UNSAT result is the solver's proof that the search for an alternative $\bar Q$ has no solution.

In [3]:
Q_identifiable = np.array([
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [0, 1, 0, 1],
], dtype=int)

result_identifiable = identify(Q_identifiable)
show_result(result_identifiable)

assert np.all(Q_identifiable.sum(axis=1) == 2)
assert result_identifiable.identifiable
assert result_identifiable.branch_label == "SAT_unsat_identifiable"
assert result_identifiable.solver_name == "glucose42"

**Identifiable:** `True`  
**Reason:** `SAT_unsat_identifiable`  
**Basis shape:** `(5, 4)`  
**SAT solver used:** `glucose42`  
**Cardinality encoding:** `prefix`

The result also records the size of the SAT instance and component timings in seconds. Timings depend on the machine and may change between runs.

In [4]:
print("SAT variables:", result_identifiable.sat_variables)
print("SAT clauses:", result_identifiable.sat_clauses)
print("Timings (seconds):")
for name, seconds in result_identifiable.timings.items():
    print(f"  {name}: {seconds:.6f}")

SAT variables: 111
SAT clauses: 266
Timings (seconds):
  basis_time: 0.000078
  identity_check_time: 0.000047
  two_col_check_time: 0.000094
  three_col_check_time: 0.000074
  sat_time: 0.007437
  algorithm_time: 0.007735


The default cardinality encoding is `prefix`. It uses the fact that every column of $H$ is nonempty, then looks for two ones in one column. Glucose 4.2 remains the solver. For a controlled comparison, PySAT's general Sinz sequential counter can be selected as follows; it solves the same mathematical problem. `cardnetwrk` and `totalizer` are also available. No encoding is fastest on every matrix.

In [5]:
result_seqcounter = identify(Q_identifiable, cardinality_encoding="seqcounter")
assert result_seqcounter.identifiable == result_identifiable.identifiable
assert result_seqcounter.solver_name == "glucose42"
show_result(result_seqcounter)

**Identifiable:** `True`  
**Reason:** `SAT_unsat_identifiable`  
**Basis shape:** `(5, 4)`  
**SAT solver used:** `glucose42`  
**Cardinality encoding:** `seqcounter`

## 3. Inspect a counterexample

For the next matrix, SAT finds $\bar Q$ and $H$ such that

$$Q=\bar Q\odot H,\qquad \bar Q\not\sim Q,$$

where $\odot$ is the Boolean product: multiply with AND and add with OR. These arrays are available as `result.counterexample` and `result.factorization`. The rows of $\bar Q$ follow the original input row order; the columns of $H$ follow the original columns of $Q$. Rows of $H$ correspond to columns of $\bar Q$.

In [6]:
Q_nonidentifiable = np.array([
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 0, 1, 1],
    [0, 1, 1, 1],
], dtype=int)

result_nonidentifiable = identify(Q_nonidentifiable)
show_result(result_nonidentifiable)

assert not result_nonidentifiable.identifiable
assert result_nonidentifiable.branch_label == "SAT_found_counterexample"
Q_bar = result_nonidentifiable.counterexample
H = result_nonidentifiable.factorization
assert Q_bar is not None and H is not None

print("Alternative Q_bar:\n", Q_bar)
print("Factor H:\n", H)
print("Boolean product Q_bar ⊙ H:\n", boolean_product(Q_bar, H))

np.testing.assert_array_equal(boolean_product(Q_bar, H), Q_nonidentifiable)
assert not equivalent_up_to_column_permutation(Q_nonidentifiable, Q_bar)
print("Verified: the Boolean product equals Q, and Q_bar is not a column permutation of Q.")

Alternative Q_bar:
 [[0 0 0 1]
 [0 0 1 0]
 [1 0 0 0]
 [0 1 1 0]
 [1 1 0 0]]
Factor H:
 [[0 1 0 1]
 [0 0 1 1]
 [1 0 1 0]
 [1 1 0 0]]
Boolean product Q_bar ⊙ H:
 [[1 1 0 0]
 [1 0 1 0]
 [0 1 0 1]
 [1 0 1 1]
 [0 1 1 1]]
Verified: the Boolean product equals Q, and Q_bar is not a column permutation of Q.


**Identifiable:** `False`  
**Reason:** `SAT_found_counterexample`  
**Basis shape:** `(5, 4)`  
**SAT solver used:** `glucose42`  
**Cardinality encoding:** `prefix`

## 4. A preprocessing decision may have no matrix certificate

Some necessary conditions can certify non-identifiability before SAT is called. The three-column check does so for this triangle matrix. The result identifies the violating columns using **zero-based Python indices**; it does not manufacture an alternative matrix.

Always check `result.counterexample is not None` before trying to inspect a certificate. Its absence does not mean the decision is unresolved.

In [7]:
Q_triangle = np.array([
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1],
], dtype=int)
result_triangle = identify(Q_triangle)

show_result(result_triangle)
print("Violating columns (zero-based):", result_triangle.violating_columns)
assert not result_triangle.identifiable
assert result_triangle.counterexample is None
assert result_triangle.factorization is None
assert result_triangle.solver_name is None

Violating columns (zero-based): (0, 1, 2)


**Identifiable:** `False`  
**Reason:** `three_column_check_failed`  
**Basis shape:** `(3, 3)`  
**SAT solver used:** `None`  
**Cardinality encoding:** `None`

## 5. Reduce and reconstruct a matrix

Basis reduction removes duplicate rows and rows generated by Boolean ORs of other rows. `identify` performs this automatically. The separate `reduce_to_basis` function is useful for inspecting the reduction.

Here we append a duplicate row and the OR of two existing rows. The stored map reconstructs the original matrix in its original row order.

In [8]:
Q_with_extra_rows = np.vstack([
    Q_nonidentifiable,
    Q_nonidentifiable[0],
    np.bitwise_or(Q_nonidentifiable[0], Q_nonidentifiable[1]),
])
reduction = reduce_to_basis(Q_with_extra_rows)

print("Original shape:", Q_with_extra_rows.shape)
print("Basis shape:", reduction.basis.shape)
print("Basis matrix:\n", reduction.basis)
print("Original rows expressed by basis-row indices:", reduction.original_to_basis)

Q_reconstructed = reconstruct_from_basis(
    reduction.basis, reduction.original_to_basis
)
np.testing.assert_array_equal(Q_reconstructed, Q_with_extra_rows)
assert identify(Q_with_extra_rows).identifiable == result_nonidentifiable.identifiable
print("Reconstruction agrees with every original row.")

Original shape: (7, 4)
Basis shape: (5, 4)
Basis matrix:
 [[0 1 0 1]
 [0 1 1 1]
 [1 0 1 0]
 [1 0 1 1]
 [1 1 0 0]]
Original rows expressed by basis-row indices: ((4,), (2,), (0,), (3,), (1,), (4,), (2, 4))
Reconstruction agrees with every original row.


## 6. Read a matrix from a CSV file

The example CSVs are in `data/examples/`. They have no header or row labels: each line is one row of $Q$. Launch the notebook from the project folder or its `notebooks/` folder so the following relative-path lookup can find them.

In [9]:
project_candidates = (Path.cwd(), Path.cwd().parent)
PROJECT_ROOT = next(
    (path for path in project_candidates
     if (path / "pyproject.toml").is_file()
     and (path / "data" / "examples" / "Q_identifiable_no_pure.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Launch this notebook from the idQ project folder or its notebooks/ folder."
    )

example_path = PROJECT_ROOT / "data" / "examples" / "Q_identifiable_no_pure.csv"
Q_from_csv = np.loadtxt(example_path, delimiter=",", dtype=int, ndmin=2)
np.testing.assert_array_equal(Q_from_csv, Q_identifiable)
show_result(identify(Q_from_csv))

**Identifiable:** `True`  
**Reason:** `SAT_unsat_identifiable`  
**Basis shape:** `(5, 4)`  
**SAT solver used:** `glucose42`  
**Cardinality encoding:** `prefix`

## 7. A small reproducible experiment

This example draws five $8\times4$ matrices with Bernoulli probability $p=0.4$. The sampler redraws zero rows, so each row follows the Bernoulli design **conditional on being nonzero**. A local NumPy generator makes the input matrices reproducible without changing global random state.

This is a usage example, not a runtime benchmark or an estimate of identifiability probabilities. It writes no files.

In [10]:
rng = np.random.default_rng(20260916)
example_results = []
for replicate in range(1, 6):
    Q_sample = sample_bernoulli_q(J=8, K=4, p=0.4, rng=rng)
    result = identify(Q_sample)
    assert np.all(Q_sample.sum(axis=1) > 0)
    example_results.append({
        "replicate": replicate,
        "basis_rows": result.basis.shape[0],
        "identifiable": result.identifiable,
        "reason": result.branch_label,
    })

table = [
    "| Replicate | Basis rows | Identifiable | Reason |",
    "|---:|---:|:---:|:---|",
]
for row in example_results:
    table.append(
        f"| {row['replicate']} | {row['basis_rows']} | {row['identifiable']} "
        f"| `{row['reason']}` |"
    )
display(Markdown("\n".join(table)))

| Replicate | Basis rows | Identifiable | Reason |
|---:|---:|:---:|:---|
| 1 | 4 | False | `two_column_check_failed` |
| 2 | 4 | False | `two_column_check_failed` |
| 3 | 4 | False | `two_column_check_failed` |
| 4 | 4 | False | `two_column_check_failed` |
| 5 | 6 | True | `SAT_unsat_identifiable` |

## 8. Optional: restrict to the maximal candidate

For a fixed $H$, an exact alternative formulation sets

$$\bar q_{j\ell}=1\quad\Longleftrightarrow\quad
h_{\ell k}\le q_{jk}\text{ for every }k.$$

This includes every row of $H$ whose support fits inside the corresponding row of $Q$. Set `maximal_candidate=True` to add this restriction. It preserves the decision for the implemented comparison class but can change the returned certificate and solving time. The default remains `False`; it is not always faster.

In [11]:
result_maximal = identify(Q_nonidentifiable, maximal_candidate=True)
show_result(result_maximal)
assert result_maximal.identifiable == result_nonidentifiable.identifiable
np.testing.assert_array_equal(
    boolean_product(result_maximal.counterexample, result_maximal.factorization),
    Q_nonidentifiable,
)

**Identifiable:** `False`  
**Reason:** `SAT_found_counterexample`  
**Basis shape:** `(5, 4)`  
**SAT solver used:** `glucose42`  
**Cardinality encoding:** `prefix`

## Where to go next

- Use `identify(your_Q)` for one matrix. Input errors raise an exception rather than returning an identifiability decision.
- Inspect `result.branch_label` to see why the algorithm stopped and `result.timings` to see where it spent time.
- Use the experiment commands in the project `README.md` for larger studies and saved CSV output.
- Use the scripts in `jobs/` for cluster submissions after configuring the account and Python environment.
- Keep original input data in `data/`; the files in `data/examples/` are small teaching examples only.

Larger $K$ does not imply a predictable solving time. Row structure, basis size, and the existence of an alternative all affect the search.